# 🟦 KuboERP · Sprint 2 en Python
**Sprint Planning, arquitectura ágil y CRUD**: proveedores, cotizaciones, órdenes de compra, inventario con stock crítico y motor de nómina.

*Automatiza • Integra • Optimiza* · Oscar Javier Bello Daza · Ingeniería de Software II · Universidad de Cundinamarca

**Cómo usarlo:** *Entorno de ejecución → Reiniciar y ejecutar todo*. El cuaderno escribe el código y las pruebas, ejecuta pytest, mide la cobertura (la Definition of Done exige más del 80 %) y corre los escenarios BDD.

## 0. Preparar el entorno

In [ ]:
!pip install -q behave coverage pytest

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
for carpeta in ("kuboerp", "tests", "features/steps"):
    os.makedirs(carpeta, exist_ok=True)
print("Carpetas listas:", sorted(os.listdir(".")))

## 1. Contratos y excepciones (SOLID · D e I)
Los servicios dependen de **protocolos**, no de una base de datos. Hoy se inyectan repositorios en memoria; en el sprint 3 se inyectará el adaptador de Azure SQL sin cambiar los servicios.

In [ ]:
%%writefile kuboerp/__init__.py
"""KuboERP · Automatiza • Integra • Optimiza.

Módulos de Compras, Logística y Almacén para una empresa de productos
y servicios eléctricos: proveedores, cotizaciones, órdenes de compra e inventario.
"""

In [ ]:
%%writefile kuboerp/excepciones.py
"""Excepciones de negocio de KuboERP (Compras, Logística, Almacén y Talento Humano)."""


class InvalidSupplierException(Exception):
    """El proveedor no cumple las reglas de registro."""


class SupplierHasActiveOrdersException(Exception):
    """El proveedor tiene órdenes de compra activas y no se puede eliminar."""


class InvalidQuotationException(Exception):
    """La cotización no cumple las reglas de registro."""


class PurchaseOrderException(Exception):
    """La orden de compra no se puede crear o no se puede cambiar de estado."""


class InvalidProductException(Exception):
    """El producto no cumple las reglas del inventario."""


class InsufficientStockException(Exception):
    """El retiro supera el stock disponible."""


class InvalidEmployeeException(Exception):
    """El empleado no cumple las reglas de registro o de contrato."""

In [ ]:
%%writefile kuboerp/repositorios.py
"""Sprint 2 · Contratos (puertos) que usan los servicios de KuboERP.

Principios SOLID aplicados:
* D (Inversión de dependencias): los servicios dependen de estos protocolos y no de una base de datos
  concreta. Hoy se inyectan repositorios en memoria; en producción se inyectará el adaptador de Azure SQL
  sin cambiar los servicios.
* I (Segregación de interfaces): cada servicio pide solo los métodos que usa. Por ejemplo, el servicio de
  proveedores solo necesita saber si un proveedor tiene órdenes activas (VerificadorOrdenesActivas), no todo
  el repositorio de órdenes.
"""
from typing import Optional, Protocol, runtime_checkable


@runtime_checkable
class RepositorioProveedores(Protocol):
    def save(self, proveedor): ...

    def find_by_nit(self, nit: str): ...

    def find_all(self) -> list: ...


@runtime_checkable
class RepositorioCotizaciones(Protocol):
    def add_request(self, codigo: str, estado: str = "ABIERTA"): ...

    def find_request(self, codigo: str): ...

    def exists_by_request_and_supplier(self, solicitud: str, nit: str) -> bool: ...

    def find_by_codigo(self, codigo: str): ...

    def find_by_request(self, solicitud: str) -> list: ...

    def save(self, cotizacion): ...


@runtime_checkable
class VerificadorOrdenesActivas(Protocol):
    def has_active_orders(self, nit: str) -> bool: ...


@runtime_checkable
class RepositorioOrdenes(VerificadorOrdenesActivas, Protocol):
    def save(self, orden): ...

    def find_by_codigo(self, codigo: str): ...

    def find_all(self) -> list: ...


@runtime_checkable
class RepositorioProductos(Protocol):
    movimientos: list

    def save(self, producto): ...

    def find_by_code(self, codigo: str): ...

    def find_all(self) -> list: ...


@runtime_checkable
class EntradaInventario(Protocol):
    """Lo único que el módulo de compras necesita del inventario: registrar una entrada."""

    def update_stock(self, product_id: str, quantity: int) -> int: ...


@runtime_checkable
class ObservadorStock(Protocol):
    """Recibe un aviso cuando un producto queda en stock crítico (patrón Observer)."""

    def notificar(self, alerta) -> None: ...


@runtime_checkable
class RepositorioEmpleados(Protocol):
    def save(self, empleado): ...

    def find_by_cedula(self, cedula: str) -> Optional[object]: ...

    def find_all(self) -> list: ...

## 2. CRUD de proveedores (EP05-US01, US02, US04)

In [ ]:
%%writefile kuboerp/proveedores.py
"""CRUD de proveedores de material eléctrico (EP05-US01, EP05-US02, EP05-US04).

Taller 1 (Sprint 2 · TDD) creó el registro y la desactivación. El taller de Sprint Planning y CRUD en
Python completa la consulta (listar) y la actualización.
"""
import re
from dataclasses import dataclass

from kuboerp.excepciones import InvalidSupplierException, SupplierHasActiveOrdersException

MSG_REGISTRADO = "Proveedor registrado exitosamente"
MSG_DESACTIVADO = "Proveedor desactivado"

_NIT = re.compile(r"^\d{9,10}$")
_CORREO = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")


@dataclass
class Supplier:
    nit: str
    nombre: str
    correo: str
    estado: str = "ACTIVO"
    presupuesto: int = 0


class InMemorySupplierRepository:
    def __init__(self):
        self._datos = {}

    def save(self, proveedor):
        self._datos[proveedor.nit] = proveedor
        return proveedor

    def find_by_nit(self, nit):
        return self._datos.get(nit)

    def find_all(self):
        return list(self._datos.values())

    def count(self):
        return len(self._datos)


class InMemoryOrderRepository:
    """Repositorio mínimo de órdenes: solo responde si un proveedor tiene órdenes activas."""

    def __init__(self, nits_con_ordenes=()):
        self._nits = set(nits_con_ordenes)

    def has_active_orders(self, nit):
        return nit in self._nits


class SupplierService:
    """Responsabilidad única: reglas de negocio de los proveedores.

    No sabe cómo se guardan los datos (recibe el repositorio por constructor) ni cómo se gestionan las
    órdenes (solo pregunta has_active_orders).
    """

    def __init__(self, repo=None, repo_ordenes=None):
        self.repo = repo if repo is not None else InMemorySupplierRepository()
        self.repo_ordenes = repo_ordenes if repo_ordenes is not None else InMemoryOrderRepository()

    # ---------- validaciones ----------
    @staticmethod
    def es_nit_valido(nit):
        return bool(nit) and bool(_NIT.match(nit))

    @staticmethod
    def es_correo_valido(correo):
        return bool(correo) and bool(_CORREO.match(correo))

    def existe_por_nit(self, nit):
        return self.repo.find_by_nit(nit) is not None

    @staticmethod
    def _validar_nombre(nombre):
        if not nombre or not nombre.strip():
            raise InvalidSupplierException("El nombre es obligatorio")

    def _validar_correo(self, correo):
        if not self.es_correo_valido(correo):
            raise InvalidSupplierException("Formato de correo inválido")

    # ---------- C ----------
    def create_supplier(self, nit, nombre, correo, presupuesto=0):
        if not self.es_nit_valido(nit):
            raise InvalidSupplierException("NIT inválido")
        self._validar_nombre(nombre)
        self._validar_correo(correo)
        if presupuesto < 0:
            raise InvalidSupplierException("El presupuesto no puede ser negativo")
        if self.existe_por_nit(nit):
            raise InvalidSupplierException("El NIT ya está registrado")
        return self.repo.save(Supplier(nit, nombre.strip(), correo, presupuesto=presupuesto))

    # ---------- R ----------
    def buscar(self, nit):
        proveedor = self.repo.find_by_nit(nit)
        if proveedor is None:
            raise InvalidSupplierException("Proveedor no encontrado")
        return proveedor

    def listar(self, estado=None):
        """Lista los proveedores ordenados por nombre; con estado filtra ACTIVO o INACTIVO."""
        proveedores = [p for p in self.repo.find_all() if estado is None or p.estado == estado]
        return sorted(proveedores, key=lambda p: p.nombre)

    def esta_activo(self, nit):
        proveedor = self.repo.find_by_nit(nit)
        return proveedor is not None and proveedor.estado == "ACTIVO"

    # ---------- U ----------
    def update_supplier(self, nit, nombre=None, correo=None, presupuesto=None):
        """Actualiza los datos editables. El NIT es la identidad del proveedor y no se cambia."""
        proveedor = self.buscar(nit)
        if nombre is not None:
            self._validar_nombre(nombre)
            proveedor.nombre = nombre.strip()
        if correo is not None:
            self._validar_correo(correo)
            proveedor.correo = correo
        if presupuesto is not None:
            if presupuesto < 0:
                raise InvalidSupplierException("El presupuesto no puede ser negativo")
            proveedor.presupuesto = presupuesto
        return self.repo.save(proveedor)

    # ---------- D (lógico) ----------
    def delete_supplier(self, nit):
        """Eliminación lógica: el proveedor pasa a INACTIVO si no tiene órdenes activas."""
        if self.repo_ordenes.has_active_orders(nit):
            raise SupplierHasActiveOrdersException(
                "No se puede eliminar un proveedor con órdenes de compra activas")
        proveedor = self.buscar(nit)
        proveedor.estado = "INACTIVO"
        self.repo.save(proveedor)
        return MSG_DESACTIVADO

## 3. CRUD de solicitudes y cotizaciones (EP05-US07)

In [ ]:
%%writefile kuboerp/cotizaciones.py
"""CRUD de solicitudes y cotizaciones de proveedores (EP05-US07).

Ciclo de vida de una cotización:  RECIBIDA → SELECCIONADA | RECHAZADA | ANULADA
Al seleccionar la cotización ganadora, las demás de la misma solicitud quedan RECHAZADAS y la solicitud
pasa a CERRADA, de modo que ya no recibe cotizaciones.
"""
from dataclasses import dataclass

from kuboerp.excepciones import InvalidQuotationException

MSG_REGISTRADA = "Cotización registrada"
MSG_ANULADA = "Cotización anulada"


@dataclass
class Quotation:
    codigo: str
    solicitud: str
    nit_proveedor: str
    monto: int
    estado: str = "RECIBIDA"


class InMemoryQuotationRepository:
    def __init__(self):
        self.solicitudes = {}      # código → estado ("ABIERTA", "CERRADA")
        self.cotizaciones = []

    def add_request(self, codigo, estado="ABIERTA"):
        self.solicitudes[codigo] = estado

    def find_request(self, codigo):
        return self.solicitudes.get(codigo)

    def exists_by_request_and_supplier(self, solicitud, nit):
        return any(c.solicitud == solicitud and c.nit_proveedor == nit and c.estado != "ANULADA"
                   for c in self.cotizaciones)

    def find_by_codigo(self, codigo):
        return next((c for c in self.cotizaciones if c.codigo == codigo), None)

    def find_by_request(self, solicitud):
        return [c for c in self.cotizaciones if c.solicitud == solicitud]

    def save(self, cotizacion):
        if cotizacion not in self.cotizaciones:
            self.cotizaciones.append(cotizacion)
        return cotizacion


class QuotationService:
    def __init__(self, repo=None):
        self.repo = repo if repo is not None else InMemoryQuotationRepository()

    # ---------- C ----------
    def abrir_solicitud(self, codigo):
        if not codigo:
            raise InvalidQuotationException("El código de la solicitud es obligatorio")
        if self.repo.find_request(codigo) is not None:
            raise InvalidQuotationException("La solicitud ya existe")
        self.repo.add_request(codigo)
        return codigo

    def registrar(self, codigo, solicitud, nit_proveedor, monto):
        if self.repo.find_request(solicitud) != "ABIERTA":
            raise InvalidQuotationException("La solicitud no está abierta")
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        if self.repo.find_by_codigo(codigo) is not None:
            raise InvalidQuotationException("El código de la cotización ya existe")
        if self.repo.exists_by_request_and_supplier(solicitud, nit_proveedor):
            raise InvalidQuotationException("El proveedor ya cotizó esta solicitud")
        return self.repo.save(Quotation(codigo, solicitud, nit_proveedor, monto))

    # ---------- R ----------
    def buscar(self, codigo):
        cotizacion = self.repo.find_by_codigo(codigo)
        if cotizacion is None:
            raise InvalidQuotationException("Cotización no encontrada")
        return cotizacion

    def listar_por_solicitud(self, solicitud):
        """Cotizaciones vigentes de la solicitud, de la más barata a la más cara."""
        vigentes = [c for c in self.repo.find_by_request(solicitud) if c.estado != "ANULADA"]
        return sorted(vigentes, key=lambda c: c.monto)

    # ---------- U ----------
    def actualizar_monto(self, codigo, monto):
        cotizacion = self._buscar_recibida(codigo)
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        cotizacion.monto = monto
        return self.repo.save(cotizacion)

    def seleccionar(self, codigo):
        """Elige la cotización ganadora, rechaza las demás y cierra la solicitud."""
        ganadora = self._buscar_recibida(codigo)
        for otra in self.repo.find_by_request(ganadora.solicitud):
            if otra.estado == "RECIBIDA":
                otra.estado = "RECHAZADA"
                self.repo.save(otra)
        ganadora.estado = "SELECCIONADA"
        self.repo.add_request(ganadora.solicitud, "CERRADA")
        return self.repo.save(ganadora)

    # ---------- D (lógico) ----------
    def anular(self, codigo):
        cotizacion = self._buscar_recibida(codigo)
        cotizacion.estado = "ANULADA"
        self.repo.save(cotizacion)
        return MSG_ANULADA

    def _buscar_recibida(self, codigo):
        cotizacion = self.buscar(codigo)
        if cotizacion.estado != "RECIBIDA":
            raise InvalidQuotationException("Solo se modifican cotizaciones en estado RECIBIDA")
        return cotizacion

## 4. CRUD de órdenes de compra con auditoría (EP05-US05, US06)

In [ ]:
%%writefile kuboerp/ordenes.py
"""CRUD de órdenes de compra (EP05-US05, EP05-US06).

Ciclo de vida:  PENDIENTE → APROBADA → RECIBIDA
                PENDIENTE | APROBADA → CANCELADA (con justificación)

* Aprobar descuenta el monto del presupuesto del proveedor (Taller 3).
* Recibir genera la entrada de cada producto en el inventario.
* Cancelar una orden aprobada devuelve el monto al presupuesto.
* Cada cambio de estado queda en el log de auditoría.
"""
from dataclasses import dataclass, field

from kuboerp.excepciones import PurchaseOrderException

ESTADOS_ACTIVOS = ("PENDIENTE", "APROBADA")


@dataclass
class LineaOrden:
    codigo_producto: str
    cantidad: int
    precio_unitario: int

    @property
    def subtotal(self):
        return self.cantidad * self.precio_unitario


@dataclass
class PurchaseOrder:
    codigo: str
    nit_proveedor: str
    monto: int
    estado: str = "PENDIENTE"
    lineas: list = field(default_factory=list)
    justificacion: str = ""


class InMemoryPurchaseOrderRepository:
    def __init__(self):
        self._datos = {}

    def save(self, orden):
        self._datos[orden.codigo] = orden
        return orden

    def find_by_codigo(self, codigo):
        return self._datos.get(codigo)

    def find_all(self):
        return list(self._datos.values())

    def has_active_orders(self, nit):
        return any(o.nit_proveedor == nit and o.estado in ESTADOS_ACTIVOS
                   for o in self._datos.values())


class PurchaseOrderService:
    """Depende de abstracciones: repositorios y un puerto de EntradaInventario (SOLID · D)."""

    def __init__(self, repo_proveedores, repo_ordenes, inventario=None):
        self.repo_proveedores = repo_proveedores
        self.repo_ordenes = repo_ordenes
        self.inventario = inventario
        self.auditoria = []        # (orden, estado anterior, estado nuevo, detalle)

    # ---------- C ----------
    def crear(self, codigo, nit_proveedor, lineas):
        if not codigo:
            raise PurchaseOrderException("El código de la orden es obligatorio")
        if self.repo_ordenes.find_by_codigo(codigo) is not None:
            raise PurchaseOrderException("El código de la orden ya existe")
        proveedor = self.repo_proveedores.find_by_nit(nit_proveedor)
        if proveedor is None or proveedor.estado != "ACTIVO":
            raise PurchaseOrderException("El proveedor no existe o está inactivo")
        if not lineas:
            raise PurchaseOrderException("La orden debe tener al menos un producto")
        if any(linea.cantidad <= 0 or linea.precio_unitario <= 0 for linea in lineas):
            raise PurchaseOrderException("Cantidad y precio unitario deben ser mayores que cero")
        monto = sum(linea.subtotal for linea in lineas)
        orden = PurchaseOrder(codigo, nit_proveedor, monto, lineas=list(lineas))
        self._auditar(orden, "", "PENDIENTE", "Orden creada")
        return self.repo_ordenes.save(orden)

    # ---------- R ----------
    def buscar(self, codigo):
        orden = self.repo_ordenes.find_by_codigo(codigo)
        if orden is None:
            raise PurchaseOrderException("Orden no encontrada")
        return orden

    def listar(self, estado=None, nit_proveedor=None):
        return [o for o in self.repo_ordenes.find_all()
                if (estado is None or o.estado == estado)
                and (nit_proveedor is None or o.nit_proveedor == nit_proveedor)]

    # ---------- U ----------
    def approve(self, codigo):
        orden = self.buscar(codigo)
        if orden.estado == "APROBADA":
            raise PurchaseOrderException("La orden ya está aprobada")
        if orden.estado != "PENDIENTE":
            raise PurchaseOrderException("Solo se aprueban órdenes pendientes")
        proveedor = self.repo_proveedores.find_by_nit(orden.nit_proveedor)
        if orden.monto > proveedor.presupuesto:
            raise PurchaseOrderException("Presupuesto insuficiente")
        proveedor.presupuesto -= orden.monto
        self.repo_proveedores.save(proveedor)
        return self._cambiar_estado(orden, "APROBADA", "Presupuesto descontado")

    def recibir(self, codigo):
        orden = self.buscar(codigo)
        if orden.estado != "APROBADA":
            raise PurchaseOrderException("Solo se reciben órdenes aprobadas")
        if self.inventario is not None:
            for linea in orden.lineas:
                self.inventario.update_stock(linea.codigo_producto, linea.cantidad)
        return self._cambiar_estado(orden, "RECIBIDA", "Entrada de inventario generada")

    # ---------- D (lógico) ----------
    def cancelar(self, codigo, justificacion):
        orden = self.buscar(codigo)
        if orden.estado not in ESTADOS_ACTIVOS:
            raise PurchaseOrderException("Solo se cancelan órdenes pendientes o aprobadas")
        if not justificacion or not justificacion.strip():
            raise PurchaseOrderException("La cancelación exige una justificación")
        if orden.estado == "APROBADA":
            proveedor = self.repo_proveedores.find_by_nit(orden.nit_proveedor)
            proveedor.presupuesto += orden.monto
            self.repo_proveedores.save(proveedor)
        orden.justificacion = justificacion.strip()
        return self._cambiar_estado(orden, "CANCELADA", orden.justificacion)

    def _cambiar_estado(self, orden, nuevo, detalle):
        self._auditar(orden, orden.estado, nuevo, detalle)
        orden.estado = nuevo
        return self.repo_ordenes.save(orden)

    def _auditar(self, orden, anterior, nuevo, detalle):
        self.auditoria.append((orden.codigo, anterior, nuevo, detalle))

## 5. Inventario y control de stock crítico (EP07) · patrón Observer (SOLID · O)

In [ ]:
%%writefile kuboerp/inventario.py
"""CRUD de inventario y control de stock crítico (EP01-US04, EP07-US02, EP07-US03, EP07-US04).

El Ejercicio 1 del taller de TDD creó update_stock() con el ciclo Rojo → Verde → Refactor. El taller de
Sprint Planning y CRUD agrega:
* stock mínimo por producto y estado CRÍTICO cuando el stock llega al mínimo o baja de él;
* alertas con el patrón Observer: el servicio avisa a quien se suscriba (tablero, correo, compras) sin
  conocerlo (SOLID · O: se agregan observadores nuevos sin modificar el servicio);
* consulta, actualización e inactivación lógica de productos.
"""
from dataclasses import dataclass

from kuboerp.excepciones import InsufficientStockException, InvalidProductException

MSG_RETIRO_EXITOSO = "Retiro exitoso"
MSG_STOCK_INSUFICIENTE = "Stock insuficiente"
MSG_INACTIVADO = "Producto inactivado"


@dataclass
class Product:
    codigo: str
    nombre: str
    categoria: str
    stock: int = 0
    stock_minimo: int = 0
    estado: str = "ACTIVO"

    @property
    def es_critico(self):
        return self.stock_minimo > 0 and self.stock <= self.stock_minimo


@dataclass(frozen=True)
class AlertaStock:
    codigo: str
    nombre: str
    stock: int
    stock_minimo: int

    @property
    def cantidad_sugerida(self):
        """Cantidad para reponer hasta el doble del stock mínimo (insumo de la orden de compra)."""
        return self.stock_minimo * 2 - self.stock


class RegistroAlertas:
    """Observador que guarda las alertas; lo usan el tablero de Almacén y las pruebas."""

    def __init__(self):
        self.alertas = []

    def notificar(self, alerta):
        self.alertas.append(alerta)


class InMemoryProductRepository:
    def __init__(self):
        self._datos = {}
        self.movimientos = []

    def save(self, producto):
        self._datos[producto.codigo] = producto
        return producto

    def find_by_code(self, codigo):
        return self._datos.get(codigo)

    def find_all(self):
        return list(self._datos.values())


class InventoryService:
    def __init__(self, repo=None):
        self.repo = repo if repo is not None else InMemoryProductRepository()
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)

    # ---------- C ----------
    def register_product(self, codigo, nombre, categoria, stock=0, stock_minimo=0):
        if not codigo:
            raise InvalidProductException("El código es obligatorio")
        if not categoria:
            raise InvalidProductException("La categoría es obligatoria")
        if stock < 0:
            raise InvalidProductException("El stock inicial no puede ser negativo")
        self._validar_minimo(stock_minimo)
        if self.repo.find_by_code(codigo) is not None:
            raise InvalidProductException("El código ya está registrado")
        return self.repo.save(Product(codigo, nombre, categoria, stock, stock_minimo))

    def update_stock(self, product_id, quantity):
        """Suma (entrada) o resta (salida, cantidad negativa) unidades al stock."""
        producto = self._buscar(product_id)
        if producto.estado != "ACTIVO":
            raise InvalidProductException("El producto está inactivo")
        nuevo_stock = producto.stock + quantity
        self._validar_stock_resultante(nuevo_stock)
        producto.stock = nuevo_stock
        self.repo.save(producto)
        self.repo.movimientos.append((product_id, quantity))
        if quantity < 0 and producto.es_critico:
            self._avisar(producto)
        return producto.stock

    def retirar(self, codigo, cantidad):
        self.update_stock(codigo, -cantidad)
        return MSG_RETIRO_EXITOSO

    # ---------- R ----------
    def stock_de(self, codigo):
        return self._buscar(codigo).stock

    def buscar(self, codigo):
        return self._buscar(codigo)

    def listar(self, categoria=None, incluir_inactivos=False):
        return sorted((p for p in self.repo.find_all()
                       if (categoria is None or p.categoria == categoria)
                       and (incluir_inactivos or p.estado == "ACTIVO")),
                      key=lambda p: p.codigo)

    def productos_criticos(self):
        """Productos activos en stock crítico, del más urgente (menor cobertura) al menos urgente."""
        criticos = [p for p in self.listar() if p.es_critico]
        return sorted(criticos, key=lambda p: p.stock / p.stock_minimo)

    # ---------- U ----------
    def actualizar_producto(self, codigo, nombre=None, categoria=None, stock_minimo=None):
        producto = self._buscar(codigo)
        if nombre is not None:
            if not nombre.strip():
                raise InvalidProductException("El nombre es obligatorio")
            producto.nombre = nombre.strip()
        if categoria is not None:
            if not categoria:
                raise InvalidProductException("La categoría es obligatoria")
            producto.categoria = categoria
        if stock_minimo is not None:
            self._validar_minimo(stock_minimo)
            producto.stock_minimo = stock_minimo
            if producto.es_critico:
                self._avisar(producto)
        return self.repo.save(producto)

    # ---------- D (lógico) ----------
    def inactivar_producto(self, codigo):
        producto = self._buscar(codigo)
        if producto.stock > 0:
            raise InvalidProductException("El producto tiene existencias sin ajustar")
        producto.estado = "INACTIVO"
        self.repo.save(producto)
        return MSG_INACTIVADO

    # ---------- apoyo ----------
    def _buscar(self, codigo):
        producto = self.repo.find_by_code(codigo)
        if producto is None:
            raise InvalidProductException("Producto no encontrado")
        return producto

    def _avisar(self, producto):
        alerta = AlertaStock(producto.codigo, producto.nombre, producto.stock, producto.stock_minimo)
        for observador in self._observadores:
            observador.notificar(alerta)

    @staticmethod
    def _validar_minimo(stock_minimo):
        if stock_minimo < 0:
            raise InvalidProductException("El stock mínimo no puede ser negativo")

    @staticmethod
    def _validar_stock_resultante(nuevo_stock):
        if nuevo_stock < 0:
            raise InsufficientStockException(MSG_STOCK_INSUFICIENTE)

## 6. Talento humano y motor de nómina (EP06) · polimorfismo (SOLID · L)

In [ ]:
%%writefile kuboerp/talento_humano.py
"""CRUD de empleados (EP06-US01, EP06-US02, EP06-US03, EP06-US05).

Retoma la jerarquía del taller de Herencia y del diagrama de clases del Sprint 1: Empleado es abstracta y
cada tipo de empleado define su propio devengado (polimorfismo). El motor de nómina trabaja con
cualquier Empleado sin preguntar su tipo (SOLID · L: toda subclase puede reemplazar a la base).
"""
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date

from kuboerp.excepciones import InvalidEmployeeException

TIPOS_CONTRATO = ("INDEFINIDO", "FIJO", "OBRA_LABOR")
MSG_DESVINCULADO = "Empleado desvinculado"


@dataclass
class Novedades:
    """Novedades del periodo (mes) que reporta el jefe inmediato."""
    dias_laborados: int = 30
    horas_extra_diurnas: float = 0
    ventas: int = 0
    bonificacion: int = 0


@dataclass
class Empleado(ABC):
    cedula: str
    nombre: str
    cargo: str
    tipo_contrato: str
    salario_base: int
    estado: str = "ACTIVO"
    historial: list = field(default_factory=list)

    #: Clase de riesgo ARL (Decreto 1295 de 1994): 1 = mínimo ... 5 = máximo.
    nivel_riesgo_arl = 1

    def salario_proporcional(self, novedades):
        return self.salario_base * novedades.dias_laborados / 30

    @abstractmethod
    def devengado_salarial(self, novedades, parametros):
        """Pagos que constituyen salario (base de aportes y prestaciones)."""

    def devengado_no_salarial(self, novedades):
        """Pagos que no constituyen salario (art. 128 del Código Sustantivo del Trabajo)."""
        return 0


@dataclass
class Tecnico(Empleado):
    """Técnico electricista de campo: cobra horas extra diurnas con recargo del 25 %."""
    nivel_riesgo_arl = 4

    def devengado_salarial(self, novedades, parametros):
        valor_hora = self.salario_base / parametros.horas_mensuales
        extras = novedades.horas_extra_diurnas * valor_hora * (1 + parametros.recargo_extra_diurna)
        return self.salario_proporcional(novedades) + extras


@dataclass
class Vendedor(Empleado):
    """Asesor comercial: salario base más comisión sobre las ventas del periodo (es salario)."""
    porcentaje_comision: float = 0.02

    def devengado_salarial(self, novedades, parametros):
        return self.salario_proporcional(novedades) + novedades.ventas * self.porcentaje_comision


@dataclass
class Gerente(Empleado):
    """Gerente: salario base y bonificación por resultados pactada como no salarial."""

    def devengado_salarial(self, novedades, parametros):
        return self.salario_proporcional(novedades)

    def devengado_no_salarial(self, novedades):
        return novedades.bonificacion


@dataclass
class Administrativo(Empleado):
    def devengado_salarial(self, novedades, parametros):
        return self.salario_proporcional(novedades)


class InMemoryEmployeeRepository:
    def __init__(self):
        self._datos = {}

    def save(self, empleado):
        self._datos[empleado.cedula] = empleado
        return empleado

    def find_by_cedula(self, cedula):
        return self._datos.get(cedula)

    def find_all(self):
        return list(self._datos.values())


class EmployeeService:
    def __init__(self, repo=None, smmlv=1_750_905):
        self.repo = repo if repo is not None else InMemoryEmployeeRepository()
        self.smmlv = smmlv

    # ---------- C ----------
    def registrar(self, empleado):
        if not empleado.cedula or not empleado.cedula.isdigit():
            raise InvalidEmployeeException("Cédula inválida")
        if not empleado.nombre or not empleado.cargo:
            raise InvalidEmployeeException("Nombre y cargo son obligatorios")
        if empleado.tipo_contrato not in TIPOS_CONTRATO:
            raise InvalidEmployeeException("Tipo de contrato inválido")
        self._validar_salario(empleado.salario_base)
        if self.repo.find_by_cedula(empleado.cedula) is not None:
            raise InvalidEmployeeException("La cédula ya está registrada")
        return self.repo.save(empleado)

    # ---------- R ----------
    def buscar(self, cedula):
        empleado = self.repo.find_by_cedula(cedula)
        if empleado is None:
            raise InvalidEmployeeException("Empleado no encontrado")
        return empleado

    def listar(self, cargo=None, tipo_contrato=None, estado="ACTIVO"):
        return [e for e in self.repo.find_all()
                if (cargo is None or e.cargo == cargo)
                and (tipo_contrato is None or e.tipo_contrato == tipo_contrato)
                and (estado is None or e.estado == estado)]

    # ---------- U ----------
    def actualizar_contrato(self, cedula, usuario, salario_base=None, cargo=None, fecha=None):
        """Cambia salario o cargo y conserva la versión anterior con fecha y usuario responsable."""
        empleado = self.buscar(cedula)
        if salario_base is not None:
            self._validar_salario(salario_base)
        empleado.historial.append({"fecha": fecha or date.today(), "usuario": usuario,
                                   "salario_base": empleado.salario_base, "cargo": empleado.cargo})
        if salario_base is not None:
            empleado.salario_base = salario_base
        if cargo is not None:
            empleado.cargo = cargo
        return self.repo.save(empleado)

    # ---------- D (lógico) ----------
    def desvincular(self, cedula):
        empleado = self.buscar(cedula)
        if empleado.estado == "INACTIVO":
            raise InvalidEmployeeException("El empleado ya está desvinculado")
        empleado.estado = "INACTIVO"
        self.repo.save(empleado)
        return MSG_DESVINCULADO

    def _validar_salario(self, salario):
        if salario < 0:
            raise InvalidEmployeeException("El salario no puede ser negativo")
        if salario < self.smmlv:
            raise InvalidEmployeeException("El salario no puede ser inferior al salario mínimo")

In [ ]:
%%writefile kuboerp/nomina.py
"""Motor de nómina mensual (EP06-US04).

Calcula, para cada empleado activo, el devengado, las deducciones, los aportes del empleador y el neto, y
genera un desprendible. Las reglas siguen la normativa colombiana vigente en 2026; los valores están en
ParametrosNomina para actualizarlos cada año sin tocar el motor (SOLID · O).

Reglas implementadas
* Auxilio de transporte para quien devenga hasta 2 SMMLV (Ley 15 de 1959), proporcional a los días.
* Deducciones del trabajador: salud 4 %, pensión 4 % y Fondo de Solidaridad Pensional 1 % desde 4 SMMLV
  (simplificación: no se modelan los puntos adicionales que la ley aplica desde 16 SMMLV).
* Pagos no salariales que superen el 40 % de la remuneración total suman a la base de aportes
  (art. 30 de la Ley 1393 de 2010).
* El ingreso base de cotización (IBC) no puede ser menor que un SMMLV proporcional a los días.
* Aportes del empleador: pensión 12 %, ARL según la clase de riesgo y caja de compensación 4 %. Salud
  (8,5 %), SENA (2 %) e ICBF (3 %) solo para quien devenga 10 SMMLV o más (exoneración, art. 114-1 del
  Estatuto Tributario).
"""
from dataclasses import dataclass, field

from kuboerp.talento_humano import Novedades

TARIFAS_ARL = {1: 0.00522, 2: 0.01044, 3: 0.02436, 4: 0.0435, 5: 0.0696}


@dataclass(frozen=True)
class ParametrosNomina:
    smmlv: int = 1_750_905                 # Decreto 1469 de 2025
    auxilio_transporte: int = 249_095      # Decreto 1470 de 2025
    horas_mensuales: int = 180             # 42 h semanales desde el 15 de julio de 2026 (Ley 2101 de 2021)
    recargo_extra_diurna: float = 0.25
    salud_trabajador: float = 0.04
    pension_trabajador: float = 0.04
    solidaridad_pensional: float = 0.01
    pension_empleador: float = 0.12
    salud_empleador: float = 0.085
    caja_compensacion: float = 0.04
    sena: float = 0.02
    icbf: float = 0.03
    limite_no_salarial: float = 0.40


@dataclass
class Desprendible:
    cedula: str
    nombre: str
    devengado_salarial: int
    auxilio_transporte: int
    devengado_no_salarial: int
    ibc: int
    salud: int
    pension: int
    solidaridad_pensional: int
    aportes_empleador: dict = field(default_factory=dict)

    @property
    def total_devengado(self):
        return self.devengado_salarial + self.auxilio_transporte + self.devengado_no_salarial

    @property
    def total_deducciones(self):
        return self.salud + self.pension + self.solidaridad_pensional

    @property
    def neto(self):
        return self.total_devengado - self.total_deducciones

    @property
    def costo_empleador(self):
        return self.total_devengado + sum(self.aportes_empleador.values())


class MotorNomina:
    def __init__(self, parametros=None):
        self.p = parametros or ParametrosNomina()

    def calcular(self, empleado, novedades=None):
        novedades = novedades or Novedades()
        p = self.p
        salarial = empleado.devengado_salarial(novedades, p)
        no_salarial = empleado.devengado_no_salarial(novedades)
        dias = novedades.dias_laborados

        auxilio = p.auxilio_transporte * dias / 30 if empleado.salario_base <= 2 * p.smmlv else 0
        ibc = self._ibc(salarial, no_salarial, dias)

        salud = ibc * p.salud_trabajador
        pension = ibc * p.pension_trabajador
        fsp = ibc * p.solidaridad_pensional if ibc >= 4 * p.smmlv else 0

        return Desprendible(
            cedula=empleado.cedula, nombre=empleado.nombre,
            devengado_salarial=round(salarial), auxilio_transporte=round(auxilio),
            devengado_no_salarial=round(no_salarial), ibc=round(ibc),
            salud=round(salud), pension=round(pension), solidaridad_pensional=round(fsp),
            aportes_empleador=self._aportes_empleador(empleado, salarial, ibc))

    def liquidar_periodo(self, empleados, novedades_por_cedula=None):
        """Desprendibles de todos los empleados activos del periodo."""
        novedades_por_cedula = novedades_por_cedula or {}
        return [self.calcular(e, novedades_por_cedula.get(e.cedula)) for e in empleados
                if e.estado == "ACTIVO"]

    @staticmethod
    def resumen(desprendibles):
        return {
            "empleados": len(desprendibles),
            "total_devengado": sum(d.total_devengado for d in desprendibles),
            "total_deducciones": sum(d.total_deducciones for d in desprendibles),
            "total_neto": sum(d.neto for d in desprendibles),
            "costo_empleador": sum(d.costo_empleador for d in desprendibles),
        }

    def _ibc(self, salarial, no_salarial, dias):
        p = self.p
        exceso = max(0.0, no_salarial - p.limite_no_salarial * (salarial + no_salarial))
        return max(salarial + exceso, p.smmlv * dias / 30)

    def _aportes_empleador(self, empleado, salarial, ibc):
        p = self.p
        aportes = {
            "pension": ibc * p.pension_empleador,
            "arl": ibc * TARIFAS_ARL[empleado.nivel_riesgo_arl],
            "caja_compensacion": ibc * p.caja_compensacion,
        }
        if salarial >= 10 * p.smmlv:
            aportes.update(salud=ibc * p.salud_empleador, sena=ibc * p.sena, icbf=ibc * p.icbf)
        return {concepto: round(valor) for concepto, valor in aportes.items()}

## 7. Pruebas unitarias
Pruebas de los talleres anteriores (regresión) y pruebas nuevas del Sprint 2.

In [ ]:
%%writefile tests/test_ep1_inventario_tdd.py
"""Ejercicio práctico 1 · TDD de InventoryService.update_stock(product_id, quantity)."""
import unittest

from kuboerp.excepciones import InsufficientStockException
from kuboerp.inventario import InventoryService


class TestUpdateStock(unittest.TestCase):
    def setUp(self):
        self.service = InventoryService()
        self.service.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40)
        self.service.register_product("TOMA-DOBLE-PT", "Toma doble con polo a tierra", "Interruptores y tomas", 2)

    def test_salida_que_deja_stock_negativo_lanza_excepcion(self):
        with self.assertRaises(InsufficientStockException):
            self.service.update_stock("TOMA-DOBLE-PT", -5)
        self.assertEqual(self.service.stock_de("TOMA-DOBLE-PT"), 2)

    def test_salida_valida_resta_del_stock(self):
        self.assertEqual(self.service.update_stock("BRK-2X30", -5), 35)

    def test_entrada_suma_al_stock(self):
        self.assertEqual(self.service.update_stock("BRK-2X30", 10), 50)


if __name__ == "__main__":
    unittest.main()

In [ ]:
%%writefile tests/test_ep2_proveedores_unittest.py
"""Ejercicio práctico 2 · Tres pruebas unittest de SupplierService.create_supplier()."""
import unittest

from kuboerp.excepciones import InvalidSupplierException
from kuboerp.proveedores import SupplierService


class TestCreateSupplier(unittest.TestCase):
    def setUp(self):
        self.service = SupplierService()

    def test_nit_valido_crea_proveedor_activo(self):
        proveedor = self.service.create_supplier(
            "900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
        self.assertEqual(proveedor.estado, "ACTIVO")

    def test_nit_con_menos_de_9_caracteres_es_rechazado(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("123", "Procables S.A.S.", "ventas@procables.com.co")

    def test_correo_sin_arroba_es_rechazado(self):
        with self.assertRaises(InvalidSupplierException):
            self.service.create_supplier("901555777", "Importadora LuzLED", "ventas.luzled.co")


if __name__ == "__main__":
    unittest.main()

In [ ]:
%%writefile tests/test_ep4_cobertura.py
"""Ejercicio práctico 4 · Pruebas agregadas para cubrir las líneas que coverage reportó sin probar.

Antes de este archivo la cobertura era 92 %. Líneas faltantes:
  cotizaciones.py 20-21, 24, 27, 30, 33-34 · repositorio en memoria (las pruebas usaban MagicMock)
  inventario.py   41, 70                    · código vacío y producto inexistente
  ordenes.py      27, 39                    · órdenes activas en el repositorio y orden inexistente
  proveedores.py  35, 45, 68, 72, 78        · conteo, repositorio de órdenes, nombre vacío,
                                              NIT duplicado y proveedor inexistente
Hallazgo: la regla "NIT único" (línea 72) solo la probaba el escenario BDD de Behave;
ninguna prueba unitaria la cubría. Con este archivo la cobertura sube a 100 %.
"""
import pytest

from kuboerp.cotizaciones import InMemoryQuotationRepository, QuotationService
from kuboerp.excepciones import (InvalidProductException, InvalidQuotationException,
                                 InvalidSupplierException, PurchaseOrderException)
from kuboerp.inventario import InventoryService
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrder, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, SupplierService


def test_cotizaciones_con_repositorio_en_memoria():
    repo = InMemoryQuotationRepository()
    repo.add_request("SC-0001")
    service = QuotationService(repo)

    cotizacion = service.registrar("COT-0001", "SC-0001", "901555777", 2300000)

    assert cotizacion.estado == "RECIBIDA"
    with pytest.raises(InvalidQuotationException, match="El proveedor ya cotizó esta solicitud"):
        service.registrar("COT-0002", "SC-0001", "901555777", 2100000)
    with pytest.raises(InvalidQuotationException, match="La solicitud no está abierta"):
        service.registrar("COT-0003", "SC-9999", "901555777", 2100000)


def test_inventario_rechaza_codigo_vacio_y_producto_inexistente():
    service = InventoryService()
    with pytest.raises(InvalidProductException, match="El código es obligatorio"):
        service.register_product("", "Pinza voltiamperimétrica", "Herramientas", 8)
    with pytest.raises(InvalidProductException, match="Producto no encontrado"):
        service.stock_de("PINZA-VA-01")


def test_ordenes_inexistentes_y_ordenes_activas_por_proveedor():
    ordenes = InMemoryPurchaseOrderRepository()
    ordenes.save(PurchaseOrder("OC-0001", "900888111", 3_000_000))
    service = PurchaseOrderService(InMemorySupplierRepository(), ordenes)

    assert ordenes.has_active_orders("900888111")
    assert not ordenes.has_active_orders("901555777")
    with pytest.raises(PurchaseOrderException, match="Orden no encontrada"):
        service.approve("OC-9999")


def test_proveedores_nombre_vacio_inexistente_y_conteo():
    service = SupplierService()
    with pytest.raises(InvalidSupplierException, match="El nombre es obligatorio"):
        service.create_supplier("900123456", "   ", "compras@electrocentro.com.co")
    with pytest.raises(InvalidSupplierException, match="Proveedor no encontrado"):
        service.delete_supplier("999999999")

    service.create_supplier("900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
    assert service.repo.count() == 1


def test_proveedor_con_nit_duplicado_no_se_crea():
    service = SupplierService()
    service.create_supplier("900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")

    with pytest.raises(InvalidSupplierException, match="El NIT ya está registrado"):
        service.create_supplier("900123456", "Schneider Electric Colombia", "ventas@schneider.com.co")
    assert service.repo.count() == 1

In [ ]:
%%writefile tests/test_sprint2_crud_compras.py
"""Sprint 2 · CRUD de proveedores, cotizaciones y órdenes de compra (pytest).

Cada prueba sigue el patrón Arrange · Act · Assert y usa repositorios en memoria (inyección de
dependencias), así que no necesita base de datos.
"""
import pytest

from kuboerp.cotizaciones import QuotationService
from kuboerp.excepciones import (InvalidQuotationException, InvalidSupplierException,
                                 PurchaseOrderException, SupplierHasActiveOrdersException)
from kuboerp.inventario import InventoryService
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, LineaOrden, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, SupplierService

NIT = "900123456"
NIT_2 = "901555777"


@pytest.fixture
def compras():
    """Proveedores, órdenes e inventario conectados como en producción, pero en memoria."""
    repo_proveedores = InMemorySupplierRepository()
    repo_ordenes = InMemoryPurchaseOrderRepository()
    proveedores = SupplierService(repo_proveedores, repo_ordenes)
    inventario = InventoryService()
    ordenes = PurchaseOrderService(repo_proveedores, repo_ordenes, inventario)

    proveedores.create_supplier(NIT, "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co",
                                presupuesto=10_000_000)
    inventario.register_product("CAB-THHN-12", "Cable THHN 12 AWG (rollo 100 m)", "Cables", 5, 10)
    inventario.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40, 15)
    return proveedores, ordenes, inventario


LINEAS = [LineaOrden("CAB-THHN-12", 20, 180_000), LineaOrden("BRK-2X30", 10, 45_000)]


# ---------------------------------------------------------------- proveedores
class TestProveedoresCrud:
    def test_listar_ordena_por_nombre_y_filtra_por_estado(self, compras):
        proveedores, _, _ = compras
        proveedores.create_supplier(NIT_2, "Andina de Iluminación", "ventas@andinaluz.com")
        proveedores.delete_supplier(NIT_2)

        assert [p.nit for p in proveedores.listar()] == [NIT_2, NIT]
        assert [p.nit for p in proveedores.listar("ACTIVO")] == [NIT]
        assert [p.nit for p in proveedores.listar("INACTIVO")] == [NIT_2]

    def test_actualizar_nombre_correo_y_presupuesto(self, compras):
        proveedores, _, _ = compras

        actualizado = proveedores.update_supplier(NIT, nombre="  Electrocentro S.A.S. ",
                                                  correo="pedidos@electrocentro.com.co",
                                                  presupuesto=12_000_000)

        assert actualizado.nombre == "Electrocentro S.A.S."
        assert actualizado.correo == "pedidos@electrocentro.com.co"
        assert proveedores.buscar(NIT).presupuesto == 12_000_000

    @pytest.mark.parametrize("cambios, mensaje", [
        ({"nombre": " "}, "El nombre es obligatorio"),
        ({"correo": "pedidos-electrocentro"}, "Formato de correo inválido"),
        ({"presupuesto": -1}, "El presupuesto no puede ser negativo"),
    ])
    def test_actualizar_con_datos_invalidos_no_cambia_el_proveedor(self, compras, cambios, mensaje):
        proveedores, _, _ = compras
        with pytest.raises(InvalidSupplierException, match=mensaje):
            proveedores.update_supplier(NIT, **cambios)
        assert proveedores.buscar(NIT).nombre == "Distribuidora Eléctrica del Centro"

    def test_no_crear_con_presupuesto_negativo(self, compras):
        proveedores, _, _ = compras
        with pytest.raises(InvalidSupplierException, match="El presupuesto no puede ser negativo"):
            proveedores.create_supplier(NIT_2, "Andina de Iluminación", "ventas@andinaluz.com", -5)

    def test_esta_activo(self, compras):
        proveedores, _, _ = compras
        assert proveedores.esta_activo(NIT)
        assert not proveedores.esta_activo("999999999")

    def test_no_desactivar_proveedor_con_orden_pendiente_del_repositorio_real(self, compras):
        proveedores, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        with pytest.raises(SupplierHasActiveOrdersException):
            proveedores.delete_supplier(NIT)


# ---------------------------------------------------------------- cotizaciones
class TestCotizacionesCrud:
    @pytest.fixture
    def servicio(self):
        servicio = QuotationService()
        servicio.abrir_solicitud("SC-0001")
        servicio.registrar("COT-0001", "SC-0001", NIT, 2_300_000)
        servicio.registrar("COT-0002", "SC-0001", NIT_2, 2_100_000)
        return servicio

    def test_abrir_solicitud_valida_codigo_y_duplicado(self, servicio):
        with pytest.raises(InvalidQuotationException, match="obligatorio"):
            servicio.abrir_solicitud("")
        with pytest.raises(InvalidQuotationException, match="La solicitud ya existe"):
            servicio.abrir_solicitud("SC-0001")

    def test_codigo_de_cotizacion_unico(self, servicio):
        with pytest.raises(InvalidQuotationException, match="El código de la cotización ya existe"):
            servicio.registrar("COT-0001", "SC-0001", "830000111", 1_900_000)

    def test_listar_por_solicitud_de_la_mas_barata_a_la_mas_cara(self, servicio):
        assert [c.codigo for c in servicio.listar_por_solicitud("SC-0001")] == ["COT-0002", "COT-0001"]

    def test_actualizar_monto(self, servicio):
        assert servicio.actualizar_monto("COT-0001", 2_000_000).monto == 2_000_000
        with pytest.raises(InvalidQuotationException, match="Monto inválido"):
            servicio.actualizar_monto("COT-0001", 0)

    def test_seleccionar_rechaza_las_demas_y_cierra_la_solicitud(self, servicio):
        ganadora = servicio.seleccionar("COT-0002")

        assert ganadora.estado == "SELECCIONADA"
        assert servicio.buscar("COT-0001").estado == "RECHAZADA"
        with pytest.raises(InvalidQuotationException, match="La solicitud no está abierta"):
            servicio.registrar("COT-0003", "SC-0001", "830000111", 1_000_000)

    def test_anular_permite_que_el_proveedor_vuelva_a_cotizar(self, servicio):
        assert servicio.anular("COT-0001") == "Cotización anulada"
        assert [c.codigo for c in servicio.listar_por_solicitud("SC-0001")] == ["COT-0002"]

        nueva = servicio.registrar("COT-0003", "SC-0001", NIT, 1_950_000)
        assert nueva.estado == "RECIBIDA"

    def test_no_modificar_cotizacion_que_no_esta_recibida(self, servicio):
        servicio.anular("COT-0001")
        with pytest.raises(InvalidQuotationException, match="Solo se modifican cotizaciones"):
            servicio.actualizar_monto("COT-0001", 1_000_000)

    def test_cotizacion_inexistente(self, servicio):
        with pytest.raises(InvalidQuotationException, match="Cotización no encontrada"):
            servicio.buscar("COT-9999")


# ---------------------------------------------------------------- órdenes de compra
class TestOrdenesCrud:
    def test_crear_calcula_el_monto_con_las_lineas(self, compras):
        _, ordenes, _ = compras

        orden = ordenes.crear("OC-0001", NIT, LINEAS)

        assert orden.estado == "PENDIENTE"
        assert orden.monto == 20 * 180_000 + 10 * 45_000      # 4.050.000
        assert ordenes.auditoria[-1] == ("OC-0001", "", "PENDIENTE", "Orden creada")

    @pytest.mark.parametrize("codigo, nit, lineas, mensaje", [
        ("", NIT, LINEAS, "El código de la orden es obligatorio"),
        ("OC-0002", "999999999", LINEAS, "El proveedor no existe o está inactivo"),
        ("OC-0002", NIT, [], "al menos un producto"),
        ("OC-0002", NIT, [LineaOrden("BRK-2X30", 0, 45_000)], "mayores que cero"),
        ("OC-0002", NIT, [LineaOrden("BRK-2X30", 3, -1)], "mayores que cero"),
    ])
    def test_crear_con_datos_invalidos(self, compras, codigo, nit, lineas, mensaje):
        _, ordenes, _ = compras
        with pytest.raises(PurchaseOrderException, match=mensaje):
            ordenes.crear(codigo, nit, lineas)

    def test_no_crear_orden_a_proveedor_inactivo_ni_con_codigo_repetido(self, compras):
        proveedores, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        with pytest.raises(PurchaseOrderException, match="El código de la orden ya existe"):
            ordenes.crear("OC-0001", NIT, LINEAS)

        proveedores.create_supplier(NIT_2, "Andina de Iluminación", "ventas@andinaluz.com")
        proveedores.delete_supplier(NIT_2)
        with pytest.raises(PurchaseOrderException, match="inactivo"):
            ordenes.crear("OC-0002", NIT_2, LINEAS)

    def test_listar_por_estado_y_proveedor(self, compras):
        _, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        ordenes.crear("OC-0002", NIT, LINEAS)
        ordenes.approve("OC-0002")

        assert [o.codigo for o in ordenes.listar("PENDIENTE")] == ["OC-0001"]
        assert len(ordenes.listar(nit_proveedor=NIT)) == 2
        assert ordenes.listar(nit_proveedor=NIT_2) == []

    def test_recibir_genera_entrada_de_inventario(self, compras):
        _, ordenes, inventario = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        ordenes.approve("OC-0001")

        orden = ordenes.recibir("OC-0001")

        assert orden.estado == "RECIBIDA"
        assert inventario.stock_de("CAB-THHN-12") == 25
        assert inventario.stock_de("BRK-2X30") == 50
        assert [a[2] for a in ordenes.auditoria] == ["PENDIENTE", "APROBADA", "RECIBIDA"]

    def test_recibir_sin_inventario_conectado_solo_cambia_el_estado(self, compras):
        proveedores, _, _ = compras
        ordenes = PurchaseOrderService(proveedores.repo, InMemoryPurchaseOrderRepository())
        ordenes.crear("OC-0001", NIT, LINEAS)
        ordenes.approve("OC-0001")
        assert ordenes.recibir("OC-0001").estado == "RECIBIDA"

    def test_no_recibir_orden_pendiente(self, compras):
        _, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        with pytest.raises(PurchaseOrderException, match="Solo se reciben órdenes aprobadas"):
            ordenes.recibir("OC-0001")

    def test_cancelar_orden_aprobada_devuelve_el_presupuesto(self, compras):
        proveedores, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        ordenes.approve("OC-0001")
        assert proveedores.buscar(NIT).presupuesto == 5_950_000

        orden = ordenes.cancelar("OC-0001", "  El proveedor no tiene el cable en bodega ")

        assert orden.estado == "CANCELADA"
        assert orden.justificacion == "El proveedor no tiene el cable en bodega"
        assert proveedores.buscar(NIT).presupuesto == 10_000_000

    def test_cancelar_exige_justificacion_y_estado_activo(self, compras):
        _, ordenes, _ = compras
        ordenes.crear("OC-0001", NIT, LINEAS)
        with pytest.raises(PurchaseOrderException, match="exige una justificación"):
            ordenes.cancelar("OC-0001", "  ")

        ordenes.cancelar("OC-0001", "Compra duplicada")
        with pytest.raises(PurchaseOrderException, match="Solo se cancelan"):
            ordenes.cancelar("OC-0001", "Otra vez")
        with pytest.raises(PurchaseOrderException, match="Solo se aprueban órdenes pendientes"):
            ordenes.approve("OC-0001")

In [ ]:
%%writefile tests/test_sprint2_inventario_critico.py
"""Sprint 2 · CRUD de inventario y control de stock crítico (unittest).

Se usa unittest (incluido en Python) para que el archivo corra igual en Google Colab, en el computador y
en GitHub Actions sin instalar nada.
"""
import unittest

from kuboerp.excepciones import InsufficientStockException, InvalidProductException
from kuboerp.inventario import AlertaStock, InventoryService, RegistroAlertas


class TestStockCritico(unittest.TestCase):
    def setUp(self):
        self.inventario = InventoryService()
        self.alertas = RegistroAlertas()
        self.inventario.suscribir(self.alertas)
        self.inventario.register_product("CAB-THHN-12", "Cable THHN 12 AWG (rollo 100 m)", "Cables", 30, 10)
        self.inventario.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40, 15)
        self.inventario.register_product("LED-6060-48W", "Panel LED 60x60 48W", "Iluminación", 8, 0)

    def test_retiro_que_deja_el_stock_en_el_minimo_genera_alerta(self):
        self.inventario.retirar("CAB-THHN-12", 20)

        self.assertEqual(self.alertas.alertas, [AlertaStock("CAB-THHN-12", "Cable THHN 12 AWG (rollo 100 m)",
                                                            10, 10)])
        self.assertTrue(self.inventario.buscar("CAB-THHN-12").es_critico)

    def test_retiro_que_no_llega_al_minimo_no_genera_alerta(self):
        self.inventario.retirar("CAB-THHN-12", 19)
        self.assertEqual(self.alertas.alertas, [])

    def test_las_entradas_no_generan_alerta(self):
        self.inventario.retirar("CAB-THHN-12", 25)        # queda en 5: alerta
        self.inventario.update_stock("CAB-THHN-12", 2)    # entrada: sigue crítico, sin alerta nueva
        self.assertEqual(len(self.alertas.alertas), 1)

    def test_producto_sin_minimo_nunca_es_critico(self):
        self.inventario.retirar("LED-6060-48W", 8)
        self.assertFalse(self.inventario.buscar("LED-6060-48W").es_critico)
        self.assertEqual(self.alertas.alertas, [])

    def test_cantidad_sugerida_repone_hasta_el_doble_del_minimo(self):
        self.inventario.retirar("CAB-THHN-12", 26)
        self.assertEqual(self.alertas.alertas[0].cantidad_sugerida, 16)   # 2 x 10 - 4

    def test_productos_criticos_del_mas_urgente_al_menos_urgente(self):
        self.inventario.retirar("CAB-THHN-12", 22)   # 8/10  = 0,80
        self.inventario.retirar("BRK-2X30", 31)      # 9/15  = 0,60
        criticos = [p.codigo for p in self.inventario.productos_criticos()]
        self.assertEqual(criticos, ["BRK-2X30", "CAB-THHN-12"])

    def test_subir_el_minimo_por_encima_del_stock_genera_alerta(self):
        self.inventario.actualizar_producto("LED-6060-48W", stock_minimo=12)
        self.assertEqual(self.alertas.alertas[0].codigo, "LED-6060-48W")

    def test_varios_observadores_reciben_la_misma_alerta(self):
        tablero = RegistroAlertas()
        self.inventario.suscribir(tablero)
        self.inventario.retirar("BRK-2X30", 30)
        self.assertEqual(self.alertas.alertas, tablero.alertas)

    def test_minimo_negativo_rechazado(self):
        with self.assertRaisesRegex(InvalidProductException, "stock mínimo no puede ser negativo"):
            self.inventario.register_product("TOMA-DOBLE-PT", "Toma doble", "Interruptores y tomas", 5, -1)


class TestInventarioCrud(unittest.TestCase):
    def setUp(self):
        self.inventario = InventoryService()
        self.inventario.register_product("CAB-THHN-12", "Cable THHN 12 AWG", "Cables", 30, 10)
        self.inventario.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 0, 5)
        self.inventario.register_product("CAB-THHN-10", "Cable THHN 10 AWG", "Cables", 12, 4)

    def test_listar_por_categoria_ordenado_por_codigo(self):
        codigos = [p.codigo for p in self.inventario.listar("Cables")]
        self.assertEqual(codigos, ["CAB-THHN-10", "CAB-THHN-12"])

    def test_actualizar_nombre_y_categoria(self):
        producto = self.inventario.actualizar_producto("CAB-THHN-12", nombre=" Cable THHN 12 AWG negro ",
                                                       categoria="Cables y alambres")
        self.assertEqual(producto.nombre, "Cable THHN 12 AWG negro")
        self.assertEqual(producto.categoria, "Cables y alambres")

    def test_actualizar_con_datos_vacios(self):
        with self.assertRaisesRegex(InvalidProductException, "El nombre es obligatorio"):
            self.inventario.actualizar_producto("CAB-THHN-12", nombre="  ")
        with self.assertRaisesRegex(InvalidProductException, "La categoría es obligatoria"):
            self.inventario.actualizar_producto("CAB-THHN-12", categoria="")

    def test_inactivar_producto_sin_existencias(self):
        self.assertEqual(self.inventario.inactivar_producto("BRK-2X30"), "Producto inactivado")
        self.assertNotIn("BRK-2X30", [p.codigo for p in self.inventario.listar()])
        self.assertIn("BRK-2X30", [p.codigo for p in self.inventario.listar(incluir_inactivos=True)])

    def test_no_inactivar_producto_con_existencias(self):
        with self.assertRaisesRegex(InvalidProductException, "existencias sin ajustar"):
            self.inventario.inactivar_producto("CAB-THHN-12")

    def test_producto_inactivo_no_recibe_movimientos(self):
        self.inventario.inactivar_producto("BRK-2X30")
        with self.assertRaisesRegex(InvalidProductException, "El producto está inactivo"):
            self.inventario.update_stock("BRK-2X30", 10)

    def test_retiro_mayor_al_stock_no_cambia_el_kardex(self):
        with self.assertRaises(InsufficientStockException):
            self.inventario.retirar("CAB-THHN-10", 13)
        self.assertEqual(self.inventario.repo.movimientos, [])


if __name__ == "__main__":
    unittest.main()

In [ ]:
%%writefile tests/test_sprint2_nomina.py
"""Sprint 2 · CRUD de empleados y motor de nómina (pytest).

Los valores esperados se calcularon a mano con los parámetros de 2026:
SMMLV $1.750.905 (Decreto 1469 de 2025) y auxilio de transporte $249.095 (Decreto 1470 de 2025).
"""
from datetime import date

import pytest

from kuboerp.excepciones import InvalidEmployeeException
from kuboerp.nomina import MotorNomina, ParametrosNomina
from kuboerp.talento_humano import (Administrativo, Empleado, EmployeeService, Gerente, Novedades,
                                    Tecnico, Vendedor)

SMMLV = 1_750_905


@pytest.fixture
def motor():
    return MotorNomina()


def tecnico():
    return Tecnico("1072345678", "Julián Rojas", "Técnico electricista", "INDEFINIDO", 2_200_000)


# ---------------------------------------------------------------- motor de nómina
def test_tecnico_con_horas_extra_y_auxilio_de_transporte(motor):
    d = motor.calcular(tecnico(), Novedades(horas_extra_diurnas=10))

    # valor hora = 2.200.000 / 180 = 12.222,22 ; 10 h x 12.222,22 x 1,25 = 152.777,78
    assert d.devengado_salarial == 2_352_778
    assert d.auxilio_transporte == 249_095
    assert d.salud == d.pension == 94_111
    assert d.solidaridad_pensional == 0
    assert d.neto == 2_352_778 + 249_095 - 188_222
    assert d.aportes_empleador == {"pension": 282_333, "arl": 102_346, "caja_compensacion": 94_111}


def test_vendedor_comision_es_salario(motor):
    vendedor = Vendedor("52987654", "Laura Gómez", "Asesora comercial", "FIJO", SMMLV,
                        porcentaje_comision=0.02)
    d = motor.calcular(vendedor, Novedades(ventas=10_000_000))
    assert d.devengado_salarial == SMMLV + 200_000
    assert d.ibc == SMMLV + 200_000


def test_gerente_bonificacion_no_salarial_y_limite_del_40(motor):
    gerente = Gerente("79123456", "Andrea Castro", "Gerente comercial", "INDEFINIDO", 12_000_000)
    d = motor.calcular(gerente, Novedades(bonificacion=9_000_000))

    # 40 % de 21.000.000 = 8.400.000 → el exceso de 600.000 suma al IBC
    assert d.devengado_no_salarial == 9_000_000
    assert d.ibc == 12_600_000
    assert d.auxilio_transporte == 0                 # devenga más de 2 SMMLV
    assert d.solidaridad_pensional == 126_000        # IBC ≥ 4 SMMLV
    assert "salud" not in d.aportes_empleador        # < 10 SMMLV: empleador exonerado


def test_salario_de_10_smmlv_o_mas_paga_salud_sena_e_icbf(motor):
    gerente = Gerente("79123456", "Andrea Castro", "Gerente general", "INDEFINIDO", 20_000_000)
    aportes = motor.calcular(gerente).aportes_empleador
    assert aportes["salud"] == 1_700_000
    assert aportes["sena"] == 400_000
    assert aportes["icbf"] == 600_000


def test_dias_laborados_proporcionales(motor):
    admin = Administrativo("1010203040", "Camila Díaz", "Auxiliar contable", "OBRA_LABOR", 2_000_000)
    d = motor.calcular(admin, Novedades(dias_laborados=15))
    assert d.devengado_salarial == 1_000_000
    assert d.auxilio_transporte == 124_548


def test_ibc_no_puede_ser_menor_que_un_smmlv(motor):
    medio_tiempo = Administrativo("1010203041", "Pedro Ruiz", "Mensajero", "FIJO", 1_500_000)
    assert motor.calcular(medio_tiempo).ibc == SMMLV


def test_parametros_configurables_sin_cambiar_el_motor():
    motor_2025 = MotorNomina(ParametrosNomina(smmlv=1_423_500, auxilio_transporte=200_000))
    assert motor_2025.calcular(tecnico()).auxilio_transporte == 200_000


def test_liquidar_periodo_omite_inactivos_y_resume(motor):
    activo = tecnico()
    inactivo = Administrativo("1010203040", "Camila Díaz", "Auxiliar contable", "FIJO", 2_000_000,
                              estado="INACTIVO")
    desprendibles = motor.liquidar_periodo([activo, inactivo], {activo.cedula: Novedades()})

    resumen = MotorNomina.resumen(desprendibles)
    assert resumen["empleados"] == 1
    assert resumen["total_neto"] == desprendibles[0].neto
    assert resumen["costo_empleador"] > resumen["total_devengado"]


def test_empleado_es_abstracto():
    with pytest.raises(TypeError):
        Empleado("1", "x", "y", "FIJO", SMMLV)


# ---------------------------------------------------------------- CRUD de empleados
@pytest.fixture
def servicio():
    servicio = EmployeeService()
    servicio.registrar(tecnico())
    return servicio


@pytest.mark.parametrize("empleado, mensaje", [
    (Tecnico("10A", "Ana", "Técnica", "FIJO", SMMLV), "Cédula inválida"),
    (Tecnico("1099", "", "Técnica", "FIJO", SMMLV), "obligatorios"),
    (Tecnico("1099", "Ana", "Técnica", "PRESTACION", SMMLV), "Tipo de contrato inválido"),
    (Tecnico("1099", "Ana", "Técnica", "FIJO", -1), "no puede ser negativo"),
    (Tecnico("1099", "Ana", "Técnica", "FIJO", 1_000_000), "inferior al salario mínimo"),
    (Tecnico("1072345678", "Ana", "Técnica", "FIJO", SMMLV), "La cédula ya está registrada"),
])
def test_registrar_con_datos_invalidos(servicio, empleado, mensaje):
    with pytest.raises(InvalidEmployeeException, match=mensaje):
        servicio.registrar(empleado)


def test_listar_por_cargo_contrato_y_estado(servicio):
    servicio.registrar(Vendedor("52987654", "Laura Gómez", "Asesora comercial", "FIJO", SMMLV))
    assert len(servicio.listar()) == 2
    assert [e.cedula for e in servicio.listar(cargo="Asesora comercial")] == ["52987654"]
    assert [e.cedula for e in servicio.listar(tipo_contrato="INDEFINIDO")] == ["1072345678"]


def test_actualizar_contrato_conserva_la_version_anterior(servicio):
    empleado = servicio.actualizar_contrato("1072345678", "jefe.th", salario_base=2_500_000,
                                            cargo="Supervisor de campo", fecha=date(2026, 11, 9))
    assert empleado.salario_base == 2_500_000
    assert empleado.cargo == "Supervisor de campo"
    assert empleado.historial == [{"fecha": date(2026, 11, 9), "usuario": "jefe.th",
                                   "salario_base": 2_200_000, "cargo": "Técnico electricista"}]


def test_actualizar_contrato_con_salario_invalido(servicio):
    with pytest.raises(InvalidEmployeeException):
        servicio.actualizar_contrato("1072345678", "jefe.th", salario_base=-5)
    assert servicio.buscar("1072345678").historial == []


def test_desvincular(servicio):
    assert servicio.desvincular("1072345678") == "Empleado desvinculado"
    assert servicio.listar() == []
    assert len(servicio.listar(estado=None)) == 1
    with pytest.raises(InvalidEmployeeException, match="ya está desvinculado"):
        servicio.desvincular("1072345678")
    with pytest.raises(InvalidEmployeeException, match="Empleado no encontrado"):
        servicio.buscar("000")

In [ ]:
%%writefile tests/test_sprint2_solid_contratos.py
"""Sprint 2 · Los adaptadores en memoria cumplen los contratos de repositorios.py (SOLID · L y D).

Si mañana se escribe el adaptador de Azure SQL, basta con agregarlo a esta lista: si cumple el contrato,
los servicios lo pueden usar sin cambios.
"""
import pytest

from kuboerp import repositorios as contratos
from kuboerp.cotizaciones import InMemoryQuotationRepository
from kuboerp.inventario import InMemoryProductRepository, InventoryService, RegistroAlertas
from kuboerp.ordenes import InMemoryPurchaseOrderRepository
from kuboerp.proveedores import InMemoryOrderRepository, InMemorySupplierRepository
from kuboerp.talento_humano import InMemoryEmployeeRepository


@pytest.mark.parametrize("implementacion, contrato", [
    (InMemorySupplierRepository(), contratos.RepositorioProveedores),
    (InMemoryQuotationRepository(), contratos.RepositorioCotizaciones),
    (InMemoryPurchaseOrderRepository(), contratos.RepositorioOrdenes),
    (InMemoryPurchaseOrderRepository(), contratos.VerificadorOrdenesActivas),
    (InMemoryOrderRepository(), contratos.VerificadorOrdenesActivas),
    (InMemoryProductRepository(), contratos.RepositorioProductos),
    (InventoryService(), contratos.EntradaInventario),
    (RegistroAlertas(), contratos.ObservadorStock),
    (InMemoryEmployeeRepository(), contratos.RepositorioEmpleados),
], ids=lambda valor: getattr(valor, "__name__", type(valor).__name__))
def test_adaptador_cumple_el_contrato(implementacion, contrato):
    assert isinstance(implementacion, contrato)


def test_repositorio_minimo_de_ordenes_no_es_un_repositorio_completo():
    """Segregación de interfaces: el verificador mínimo solo sirve donde se pide VerificadorOrdenesActivas."""
    assert not isinstance(InMemoryOrderRepository(), contratos.RepositorioOrdenes)

In [ ]:
%%writefile tests/test_taller1_proveedores.py
"""Taller 1 · Eliminar (desactivar) proveedores, con el repositorio de órdenes simulado."""
import unittest
from unittest.mock import MagicMock

import pytest

from kuboerp.excepciones import SupplierHasActiveOrdersException
from kuboerp.proveedores import SupplierService


class TestEliminarProveedor(unittest.TestCase):
    def test_no_deberia_eliminar_con_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = True
        service = SupplierService(repo_ordenes=repo_ordenes)
        service.create_supplier("900888111", "Procables S.A.S.", "ventas@procables.com.co")

        with self.assertRaises(SupplierHasActiveOrdersException):
            service.delete_supplier("900888111")
        self.assertEqual(service.buscar("900888111").estado, "ACTIVO")

    def test_deberia_desactivar_proveedor_sin_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = False
        service = SupplierService(repo_ordenes=repo_ordenes)
        service.create_supplier("901555777", "Importadora LuzLED", "ventas@importadoraluzled.com")

        mensaje = service.delete_supplier("901555777")

        self.assertEqual(mensaje, "Proveedor desactivado")
        self.assertEqual(service.buscar("901555777").estado, "INACTIVO")
        repo_ordenes.has_active_orders.assert_called_once_with("901555777")


# Ejemplo de la sección 3: fixture + parametrización + mock con pytest
@pytest.fixture
def supplier_service():
    repo = MagicMock()
    return SupplierService(repo)


@pytest.mark.parametrize("nit,esperado", [
    ("900123456", True),
    ("123", False),
    ("", False),
])
def test_validar_nit(supplier_service, nit, esperado):
    assert supplier_service.es_nit_valido(nit) == esperado

In [ ]:
%%writefile tests/test_taller2_cotizaciones.py
"""Taller 2 · Validar el monto de una cotización con @pytest.mark.parametrize."""
from unittest.mock import MagicMock

import pytest

from kuboerp.cotizaciones import QuotationService
from kuboerp.excepciones import InvalidQuotationException


@pytest.fixture
def repo():
    repo = MagicMock()
    repo.find_request.return_value = "ABIERTA"
    repo.exists_by_request_and_supplier.return_value = False
    repo.find_by_codigo.return_value = None   # Sprint 2: el código de la cotización debe ser único
    repo.save.side_effect = lambda cotizacion: cotizacion
    return repo


@pytest.mark.parametrize("monto,resultado", [
    (2500000, "Cotización registrada"),
    (0, "Monto inválido"),
    (-1000, "Monto inválido"),
], ids=["monto_2500000", "monto_0", "monto_negativo_1000"])
def test_validar_monto_de_cotizacion(repo, monto, resultado):
    service = QuotationService(repo)

    if monto > 0:
        cotizacion = service.registrar("COT-0001", "SC-0001", "901555777", monto)
        assert cotizacion.estado == "RECIBIDA"
        assert resultado == "Cotización registrada"
        repo.save.assert_called_once()
    else:
        with pytest.raises(InvalidQuotationException, match=resultado):
            service.registrar("COT-0001", "SC-0001", "901555777", monto)
        repo.save.assert_not_called()


def test_rechaza_cotizacion_sobre_solicitud_cerrada(repo):
    repo.find_request.return_value = "CERRADA"

    with pytest.raises(InvalidQuotationException, match="La solicitud no está abierta"):
        QuotationService(repo).registrar("COT-0002", "SC-0002", "901555777", 2300000)
    repo.save.assert_not_called()


def test_rechaza_segunda_cotizacion_del_mismo_proveedor(repo):
    repo.exists_by_request_and_supplier.return_value = True

    with pytest.raises(InvalidQuotationException, match="El proveedor ya cotizó esta solicitud"):
        QuotationService(repo).registrar("COT-0003", "SC-0001", "901555777", 2300000)
    repo.save.assert_not_called()

In [ ]:
%%writefile tests/test_taller3_ordenes.py
"""Taller 3 · Aprobación de órdenes de compra (pytest con fixtures)."""
import pytest

from kuboerp.excepciones import PurchaseOrderException
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrder, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, Supplier

NIT = "900123456"


@pytest.fixture
def repos():
    proveedores = InMemorySupplierRepository()
    proveedores.save(Supplier(NIT, "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co",
                              presupuesto=10_000_000))
    return proveedores, InMemoryPurchaseOrderRepository()


def test_aprobar_orden_dentro_del_presupuesto(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0001", NIT, 5_000_000))

    orden = PurchaseOrderService(proveedores, ordenes).approve("OC-0001")

    assert orden.estado == "APROBADA"
    assert proveedores.find_by_nit(NIT).presupuesto == 5_000_000


def test_rechazar_orden_que_excede_el_presupuesto(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0002", NIT, 15_000_000))

    with pytest.raises(PurchaseOrderException, match="Presupuesto insuficiente"):
        PurchaseOrderService(proveedores, ordenes).approve("OC-0002")
    assert ordenes.find_by_codigo("OC-0002").estado == "PENDIENTE"
    assert proveedores.find_by_nit(NIT).presupuesto == 10_000_000


def test_no_volver_a_aprobar_una_orden_aprobada(repos):
    proveedores, ordenes = repos
    ordenes.save(PurchaseOrder("OC-0003", NIT, 1_000_000, "APROBADA"))

    with pytest.raises(PurchaseOrderException, match="La orden ya está aprobada"):
        PurchaseOrderService(proveedores, ordenes).approve("OC-0003")

In [ ]:
%%writefile tests/test_taller4_inventario.py
"""Taller 4 · Retiro de inventario recorriendo los casos del esquema con subTest."""
import unittest

from kuboerp.excepciones import InsufficientStockException, InvalidProductException
from kuboerp.inventario import InventoryService

CASOS = [
    # codigo,          stock, retiro, final, mensaje
    ("CAB-THHN-12",    10,    3,      7,     "Retiro exitoso"),
    ("BRK-2X30",       5,     5,      0,     "Retiro exitoso"),
    ("TOMA-DOBLE-PT",  2,     5,      2,     "Stock insuficiente"),
]


class TestRetiroInventario(unittest.TestCase):
    def test_retiro_de_inventario(self):
        for codigo, stock, retiro, final, mensaje in CASOS:
            with self.subTest(codigo=codigo, stock=stock, retiro=retiro):
                service = InventoryService()
                service.register_product(codigo, f"Producto {codigo}", "Material eléctrico", stock)

                if retiro > stock:
                    with self.assertRaises(InsufficientStockException) as error:
                        service.retirar(codigo, retiro)
                    self.assertEqual(str(error.exception), mensaje)
                else:
                    self.assertEqual(service.retirar(codigo, retiro), mensaje)
                self.assertEqual(service.stock_de(codigo), final)

    def test_no_registra_codigo_duplicado(self):
        service = InventoryService()
        service.register_product("BRK-2X30", "Breaker 2x30A", "Breakers y tableros", 40)
        with self.assertRaises(InvalidProductException):
            service.register_product("BRK-2X30", "Otro breaker", "Breakers y tableros", 5)

    def test_no_registra_sin_categoria_ni_con_stock_negativo(self):
        service = InventoryService()
        with self.assertRaises(InvalidProductException):
            service.register_product("LED-6060-48W", "Panel LED 60x60 48W", "", 10)
        with self.assertRaises(InvalidProductException):
            service.register_product("LED-6060-48W", "Panel LED 60x60 48W", "Iluminación", -1)


if __name__ == "__main__":
    unittest.main()

In [ ]:
%%writefile .coveragerc
[run]
source = kuboerp

[report]
show_missing = true
fail_under = 80
# Los métodos de los protocolos (repositorios.py) solo declaran el contrato: su cuerpo "..." nunca se ejecuta.
exclude_also =
    def .*: \.\.\.$

### Ejecutar todas las pruebas

In [ ]:
!{sys.executable} -m pytest -v

### Cobertura (DoD: mayor que 80 %)

In [ ]:
!{sys.executable} -m coverage run -m pytest -q
!{sys.executable} -m coverage report

## 8. Escenarios BDD en español (Behave)

In [ ]:
%%writefile features/cotizaciones.feature
# language: es
# Ejercicio práctico 3 y Taller 2 · Cotizaciones de proveedores (Mockup 3 · Compras)
Característica: Registro de cotizaciones de material eléctrico
  Como jefe de compras
  Quiero registrar las cotizaciones que envían los proveedores
  Para comparar precios antes de generar la orden de compra

  Antecedentes:
    Dado que existe el proveedor "Importadora LuzLED" con NIT "901555777"
    Y que existe una solicitud "SC-0001" en estado "ABIERTA"

  Escenario: Registrar la cotización de un proveedor
    Cuando el proveedor con NIT "901555777" cotiza la solicitud "SC-0001" por 2300000 con el código "COT-0001"
    Entonces la cotización "COT-0001" queda en estado "RECIBIDA"
    Y el sistema responde "Cotización registrada"

  Esquema del escenario: Validar monto de cotización
    Cuando registro una cotización por <monto>
    Entonces el sistema responde "<resultado>"

    Ejemplos:
      | monto   | resultado             |
      | 2500000 | Cotización registrada |
      | 0       | Monto inválido        |
      | -1000   | Monto inválido        |

  Escenario: Rechazar una segunda cotización del mismo proveedor
    Dado que el proveedor con NIT "901555777" ya cotizó la solicitud "SC-0001"
    Cuando registro una cotización por 2500000
    Entonces el sistema responde "El proveedor ya cotizó esta solicitud"

In [ ]:
%%writefile features/inventario.feature
# language: es
# Taller 4 y Ejercicio práctico 1 · Inventario de la Bodega Principal (Mockup 3 · Almacén)
Característica: Entradas y retiros de material eléctrico
  Como bodeguero de la Bodega Principal
  Quiero registrar entradas y retiros de productos
  Para mantener el stock de seguridad de cables, breakers y luminarias

  Esquema del escenario: Retiro de inventario
    Dado que el producto "<codigo>" tiene stock <stock>
    Cuando se retiran <retiro> unidades
    Entonces el stock final es <final>
    Y el sistema responde "<mensaje>"

    Ejemplos:
      | codigo        | stock | retiro | final | mensaje            |
      | CAB-THHN-12   | 10    | 3      | 7     | Retiro exitoso     |
      | BRK-2X30      | 5     | 5      | 0     | Retiro exitoso     |
      | TOMA-DOBLE-PT | 2     | 5      | 2     | Stock insuficiente |

  Escenario: Registrar una entrada de mercancía
    Dado que el producto "BRK-2X30" tiene stock 40
    Cuando se registra una entrada de 10 unidades
    Entonces el stock final es 50

  Escenario: Rechazar una salida que deja el stock negativo
    Dado que el producto "TOMA-DOBLE-PT" tiene stock 2
    Cuando se retiran 5 unidades
    Entonces el sistema muestra "Stock insuficiente"
    Y el stock final es 2

In [ ]:
%%writefile features/nomina.feature
# language: es
# Sprint 2 · Motor de nómina (EP06-US04) con parámetros de 2026
Característica: Cálculo de la nómina mensual
  Como jefe de talento humano
  Quiero calcular la nómina del periodo
  Para pagar a tiempo y con los descuentos de ley

  Esquema del escenario: Auxilio de transporte según el salario
    Dado un empleado administrativo con salario base <salario>
    Cuando se calcula la nómina de 30 días
    Entonces el auxilio de transporte es <auxilio>
    Y el descuento de salud es <salud>

    Ejemplos:
      | salario   | auxilio | salud   |
      | 1750905   | 249095  | 70036   |
      | 3501810   | 249095  | 140072  |
      | 3600000   | 0       | 144000  |

In [ ]:
%%writefile features/ordenes_compra.feature
# language: es
# Taller 3 · Aprobación de Órdenes de Compra (Mockup 3 · Compras)
Característica: Aprobación de órdenes de compra contra el presupuesto del proveedor
  Como jefe de compras
  Quiero aprobar las órdenes de compra sin exceder el presupuesto de cada proveedor
  Para controlar el gasto en material eléctrico

  Antecedentes:
    Dado que el proveedor "Distribuidora Eléctrica del Centro" tiene presupuesto disponible de $10.000.000

  Escenario: Aprobar orden dentro del presupuesto
    Dado que existe una orden "OC-0001" por $5.000.000 en estado "PENDIENTE"
    Cuando el jefe de compras aprueba la orden
    Entonces la orden cambia a "APROBADA"
    Y el presupuesto del proveedor se reduce a $5.000.000

  Escenario: Rechazar orden que excede el presupuesto
    Dado que existe una orden "OC-0002" por $15.000.000 en estado "PENDIENTE"
    Cuando el jefe de compras aprueba la orden
    Entonces el sistema muestra "Presupuesto insuficiente"
    Y la orden permanece en "PENDIENTE"
    Y el presupuesto disponible del proveedor sigue en $10.000.000

  Escenario: No volver a aprobar una orden ya aprobada
    Dado que existe una orden "OC-0003" por $1.000.000 en estado "APROBADA"
    Cuando el jefe de compras aprueba la orden
    Entonces el sistema muestra "La orden ya está aprobada"

In [ ]:
%%writefile features/proveedores.feature
# language: es
# Taller 1 · CRUD de Proveedores (Mockup 3 · Compras, Logística y Almacén)
Característica: Gestión de proveedores de material eléctrico en KuboERP
  Como jefe de compras de la empresa
  Quiero registrar proveedores con NIT y correo válidos
  Para mantener actualizada la base de fabricantes, mayoristas e importadores

  Escenario: Registrar un proveedor válido
    Dado que no existe un proveedor con NIT "900123456"
    Cuando registro el proveedor con NIT "900123456", nombre "Distribuidora Eléctrica del Centro" y correo "compras@electrocentro.com.co"
    Entonces el proveedor se crea con estado "ACTIVO"
    Y el sistema muestra "Proveedor registrado exitosamente"

  Escenario: Rechazar proveedor con NIT duplicado
    Dado que ya existe un proveedor con NIT "900123456"
    Cuando intento registrar otro proveedor con el mismo NIT
    Entonces el sistema muestra "El NIT ya está registrado"
    Y no se crea un nuevo proveedor

  Esquema del escenario: Validar formatos de correo
    Cuando registro un proveedor con correo "<correo>"
    Entonces el sistema responde "<resultado>"

    Ejemplos:
      | correo                       | resultado                  |
      | ventas@importadoraluzled.com | Correo válido              |
      | sin-arroba.com               | Formato de correo inválido |
      | usuario@                     | Formato de correo inválido |

  Escenario: Eliminar proveedor sin órdenes activas
    Dado que existe un proveedor "Importadora LuzLED" con NIT "901555777" sin órdenes de compra activas
    Cuando solicito eliminar el proveedor
    Entonces el proveedor cambia a estado "INACTIVO"
    Y el sistema confirma "Proveedor desactivado"

  Escenario: No eliminar proveedor con órdenes activas
    Dado que existe un proveedor "Procables S.A.S." con NIT "900888111" y la orden de compra "OC-0001" en estado "PENDIENTE"
    Cuando solicito eliminar el proveedor
    Entonces el sistema muestra "No se puede eliminar un proveedor con órdenes de compra activas"
    Y el proveedor sigue en estado "ACTIVO"

In [ ]:
%%writefile features/stock_critico.feature
# language: es
# Sprint 2 · Control de stock crítico (EP07-US02 · política P04 del Event Storming)
Característica: Alertas de stock crítico en la Bodega Principal
  Como administrador de inventario
  Quiero recibir una alerta cuando un material llega a su stock mínimo
  Para generar la orden de compra antes de quedarme sin existencias

  Escenario: Un retiro deja el cable en su stock mínimo
    Dado que el producto "CAB-THHN-12" tiene stock 30 y stock mínimo 10
    Cuando se retiran 20 unidades
    Entonces se genera una alerta de stock crítico para "CAB-THHN-12"
    Y la cantidad sugerida para reponer es 10

  Escenario: Un retiro que no llega al mínimo no genera alerta
    Dado que el producto "BRK-2X30" tiene stock 40 y stock mínimo 15
    Cuando se retiran 5 unidades
    Entonces no se genera ninguna alerta

In [ ]:
%%writefile features/environment.py
"""Behave · prepara servicios nuevos (en memoria) antes de cada escenario."""
from kuboerp.cotizaciones import InMemoryQuotationRepository, QuotationService
from kuboerp.inventario import InventoryService
from kuboerp.ordenes import InMemoryPurchaseOrderRepository, PurchaseOrderService
from kuboerp.proveedores import InMemorySupplierRepository, SupplierService


def before_scenario(context, scenario):
    context.repo_proveedores = InMemorySupplierRepository()
    context.repo_ordenes = InMemoryPurchaseOrderRepository()
    context.repo_cotizaciones = InMemoryQuotationRepository()

    context.proveedores = SupplierService(context.repo_proveedores, context.repo_ordenes)
    context.ordenes = PurchaseOrderService(context.repo_proveedores, context.repo_ordenes)
    context.cotizaciones = QuotationService(context.repo_cotizaciones)
    context.inventario = InventoryService()

    context.mensaje = None
    context.excepcion = None

    def ejecutar(accion):
        """Ejecuta el paso @when y guarda el mensaje de éxito o el de la excepción."""
        try:
            context.mensaje = accion()
            context.excepcion = None
        except Exception as error:  # noqa: BLE001 - el escenario verifica el mensaje
            context.excepcion = error
            context.mensaje = str(error)

    context.ejecutar = ejecutar

In [ ]:
%%writefile features/steps/cotizaciones_steps.py
"""Ejercicio práctico 3 y Taller 2 · Step definitions de cotizaciones."""
from behave import given, then, when

from kuboerp.cotizaciones import MSG_REGISTRADA


@given('que existe el proveedor "{nombre}" con NIT "{nit}"')
def step_existe_proveedor(context, nombre, nit):
    context.proveedores.create_supplier(nit, nombre, "ventas@importadoraluzled.com")
    context.nit = nit


@given('que existe una solicitud "{codigo}" en estado "{estado}"')
def step_existe_solicitud(context, codigo, estado):
    context.repo_cotizaciones.add_request(codigo, estado)
    context.solicitud = codigo


@when('el proveedor con NIT "{nit}" cotiza la solicitud "{solicitud}" por {monto:d} con el código "{codigo}"')
def step_cotiza(context, nit, solicitud, monto, codigo):
    def accion():
        context.cotizacion = context.cotizaciones.registrar(codigo, solicitud, nit, monto)
        return MSG_REGISTRADA
    context.ejecutar(accion)


@then('la cotización "{codigo}" queda en estado "{estado}"')
def step_cotizacion_en_estado(context, codigo, estado):
    assert context.cotizacion.codigo == codigo
    assert context.cotizacion.estado == estado


@when('registro una cotización por {monto:d}')
def step_registro_cotizacion(context, monto):
    def accion():
        context.cotizaciones.registrar("COT-0002", context.solicitud, context.nit, monto)
        return MSG_REGISTRADA
    context.ejecutar(accion)


@given('que el proveedor con NIT "{nit}" ya cotizó la solicitud "{solicitud}"')
def step_ya_cotizo(context, nit, solicitud):
    context.cotizaciones.registrar("COT-0001", solicitud, nit, 2300000)

In [ ]:
%%writefile features/steps/inventario_steps.py
"""Taller 4 · Step definitions de inventario."""
from behave import given, then, when

CATALOGO = {
    "CAB-THHN-12": ("Cable THHN 12 AWG (rollo 100 m)", "Cables"),
    "BRK-2X30": ("Breaker 2x30A", "Breakers y tableros"),
    "LED-6060-48W": ("Panel LED 60x60 48W", "Iluminación"),
    "TOMA-DOBLE-PT": ("Toma doble con polo a tierra", "Interruptores y tomas"),
}


@given('que el producto "{codigo}" tiene stock {stock:d}')
def step_producto_con_stock(context, codigo, stock):
    nombre, categoria = CATALOGO.get(codigo, (codigo, "Material eléctrico"))
    context.inventario.register_product(codigo, nombre, categoria, stock)
    context.codigo = codigo


@when('se retiran {cantidad:d} unidades')
def step_se_retiran(context, cantidad):
    context.ejecutar(lambda: context.inventario.retirar(context.codigo, cantidad))


@when('se registra una entrada de {cantidad:d} unidades')
def step_entrada(context, cantidad):
    context.ejecutar(lambda: context.inventario.update_stock(context.codigo, cantidad))


@then('el stock final es {final:d}')
def step_stock_final(context, final):
    assert context.inventario.stock_de(context.codigo) == final

In [ ]:
%%writefile features/steps/mensajes_steps.py
"""Pasos comunes: verifican el mensaje que muestra KuboERP."""
from behave import then


@then('el sistema muestra "{mensaje}"')
def step_sistema_muestra(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"


@then('el sistema responde "{mensaje}"')
def step_sistema_responde(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"


@then('el sistema confirma "{mensaje}"')
def step_sistema_confirma(context, mensaje):
    assert context.mensaje == mensaje, f"Se esperaba '{mensaje}' y se obtuvo '{context.mensaje}'"

In [ ]:
%%writefile features/steps/ordenes_steps.py
"""Taller 3 · Step definitions de la aprobación de órdenes de compra."""
import parse
from behave import given, register_type, then, when

from kuboerp.ordenes import PurchaseOrder
from kuboerp.proveedores import Supplier

NIT_PROVEEDOR = "900123456"


@parse.with_pattern(r"\$[\d.]+")
def parse_pesos(texto):
    """Convierte "$10.000.000" en 10000000."""
    return int(texto.replace("$", "").replace(".", ""))


register_type(Pesos=parse_pesos)


@given('que el proveedor "{nombre}" tiene presupuesto disponible de {presupuesto:Pesos}')
def step_presupuesto_proveedor(context, nombre, presupuesto):
    context.repo_proveedores.save(
        Supplier(NIT_PROVEEDOR, nombre, "compras@electrocentro.com.co", presupuesto=presupuesto))


@given('que existe una orden "{codigo}" por {monto:Pesos} en estado "{estado}"')
def step_existe_orden(context, codigo, monto, estado):
    context.repo_ordenes.save(PurchaseOrder(codigo, NIT_PROVEEDOR, monto, estado))
    context.codigo_orden = codigo


@when('el jefe de compras aprueba la orden')
def step_aprueba_orden(context):
    def accion():
        context.ordenes.approve(context.codigo_orden)
        return "Orden aprobada"
    context.ejecutar(accion)


@then('la orden cambia a "{estado}"')
def step_orden_cambia(context, estado):
    assert context.repo_ordenes.find_by_codigo(context.codigo_orden).estado == estado


@then('la orden permanece en "{estado}"')
def step_orden_permanece(context, estado):
    assert context.repo_ordenes.find_by_codigo(context.codigo_orden).estado == estado


@then('el presupuesto del proveedor se reduce a {esperado:Pesos}')
def step_presupuesto_reducido(context, esperado):
    # Se consulta el presupuesto actualizado en el repositorio y se compara con el esperado.
    actual = context.repo_proveedores.find_by_nit(NIT_PROVEEDOR).presupuesto
    assert actual == esperado, f"Presupuesto esperado {esperado}, actual {actual}"


@then('el presupuesto disponible del proveedor sigue en {esperado:Pesos}')
def step_presupuesto_sin_cambio(context, esperado):
    actual = context.repo_proveedores.find_by_nit(NIT_PROVEEDOR).presupuesto
    assert actual == esperado, f"Presupuesto esperado {esperado}, actual {actual}"

In [ ]:
%%writefile features/steps/proveedores_steps.py
"""Taller 1 · Step definitions de proveedores."""
from behave import given, then, when

from kuboerp.ordenes import PurchaseOrder
from kuboerp.proveedores import MSG_REGISTRADO


@given('que no existe un proveedor con NIT "{nit}"')
def step_no_existe_proveedor(context, nit):
    assert not context.proveedores.existe_por_nit(nit)


@when('registro el proveedor con NIT "{nit}", nombre "{nombre}" y correo "{correo}"')
def step_registro_proveedor(context, nit, nombre, correo):
    def accion():
        context.proveedor = context.proveedores.create_supplier(nit, nombre, correo)
        return MSG_REGISTRADO
    context.ejecutar(accion)


@then('el proveedor se crea con estado "{estado}"')
def step_proveedor_creado(context, estado):
    assert context.proveedor is not None
    assert context.proveedor.estado == estado


@given('que ya existe un proveedor con NIT "{nit}"')
def step_ya_existe_proveedor(context, nit):
    context.proveedores.create_supplier(nit, "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co")
    context.nit = nit
    context.proveedores_antes = context.repo_proveedores.count()


@when('intento registrar otro proveedor con el mismo NIT')
def step_registro_duplicado(context):
    context.ejecutar(lambda: context.proveedores.create_supplier(
        context.nit, "Schneider Electric Colombia", "ventas@schneider.com.co"))


@then('no se crea un nuevo proveedor')
def step_no_se_crea(context):
    assert context.repo_proveedores.count() == context.proveedores_antes


@when('registro un proveedor con correo "{correo}"')
def step_registro_con_correo(context, correo):
    def accion():
        context.proveedores.create_supplier("901555777", "Importadora LuzLED", correo)
        return "Correo válido"
    context.ejecutar(accion)


@given('que existe un proveedor "{nombre}" con NIT "{nit}" sin órdenes de compra activas')
def step_proveedor_sin_ordenes(context, nombre, nit):
    context.proveedores.create_supplier(nit, nombre, "contacto@proveedor.com.co")
    context.nit = nit
    assert not context.repo_ordenes.has_active_orders(nit)


@given('que existe un proveedor "{nombre}" con NIT "{nit}" y la orden de compra "{orden}" en estado "{estado}"')
def step_proveedor_con_orden(context, nombre, nit, orden, estado):
    context.proveedores.create_supplier(nit, nombre, "contacto@proveedor.com.co")
    context.repo_ordenes.save(PurchaseOrder(orden, nit, 3_000_000, estado))
    context.nit = nit


@when('solicito eliminar el proveedor')
def step_eliminar_proveedor(context):
    context.ejecutar(lambda: context.proveedores.delete_supplier(context.nit))


@then('el proveedor cambia a estado "{estado}"')
def step_proveedor_cambia_estado(context, estado):
    assert context.proveedores.buscar(context.nit).estado == estado


@then('el proveedor sigue en estado "{estado}"')
def step_proveedor_sigue_estado(context, estado):
    assert context.proveedores.buscar(context.nit).estado == estado

In [ ]:
%%writefile features/steps/sprint2_steps.py
"""Sprint 2 · Step definitions de stock crítico y nómina."""
from behave import given, then, when

from kuboerp.inventario import RegistroAlertas
from kuboerp.nomina import MotorNomina
from kuboerp.talento_humano import Administrativo, Novedades


@given('que el producto "{codigo}" tiene stock {stock:d} y stock mínimo {minimo:d}')
def step_producto_con_minimo(context, codigo, stock, minimo):
    context.alertas = RegistroAlertas()
    context.inventario.suscribir(context.alertas)
    context.inventario.register_product(codigo, codigo, "Material eléctrico", stock, minimo)
    context.codigo = codigo


@then('se genera una alerta de stock crítico para "{codigo}"')
def step_alerta(context, codigo):
    assert [a.codigo for a in context.alertas.alertas] == [codigo]


@then('la cantidad sugerida para reponer es {cantidad:d}')
def step_cantidad_sugerida(context, cantidad):
    assert context.alertas.alertas[0].cantidad_sugerida == cantidad


@then('no se genera ninguna alerta')
def step_sin_alerta(context):
    assert context.alertas.alertas == []


@given('un empleado administrativo con salario base {salario:d}')
def step_empleado(context, salario):
    context.empleado = Administrativo("1010203040", "Camila Díaz", "Auxiliar", "FIJO", salario)


@when('se calcula la nómina de {dias:d} días')
def step_calcular(context, dias):
    context.desprendible = MotorNomina().calcular(context.empleado, Novedades(dias_laborados=dias))


@then('el auxilio de transporte es {valor:d}')
def step_auxilio(context, valor):
    assert context.desprendible.auxilio_transporte == valor


@then('el descuento de salud es {valor:d}')
def step_salud(context, valor):
    assert context.desprendible.salud == valor

In [ ]:
!{sys.executable} -m behave --format progress

## 9. Demostración del incremento

In [ ]:
import importlib, kuboerp
for m in ("excepciones","repositorios","proveedores","cotizaciones","ordenes","inventario","talento_humano","nomina"):
    importlib.reload(importlib.import_module(f"kuboerp.{m}"))

from kuboerp.proveedores import SupplierService, InMemorySupplierRepository
from kuboerp.ordenes import PurchaseOrderService, InMemoryPurchaseOrderRepository, LineaOrden
from kuboerp.inventario import InventoryService, RegistroAlertas

repo_p, repo_o = InMemorySupplierRepository(), InMemoryPurchaseOrderRepository()
proveedores = SupplierService(repo_p, repo_o)
inventario = InventoryService(); tablero = RegistroAlertas(); inventario.suscribir(tablero)
ordenes = PurchaseOrderService(repo_p, repo_o, inventario)

proveedores.create_supplier("900123456", "Distribuidora Eléctrica del Centro", "compras@electrocentro.com.co", 10_000_000)
inventario.register_product("CAB-THHN-12", "Cable THHN 12 AWG (rollo 100 m)", "Cables", 30, 10)

inventario.retirar("CAB-THHN-12", 24)                       # queda en 6: stock crítico
alerta = tablero.alertas[-1]
print(f"⚠️  {alerta.nombre}: stock {alerta.stock} ≤ mínimo {alerta.stock_minimo}; reponer {alerta.cantidad_sugerida}")

ordenes.crear("OC-0001", "900123456", [LineaOrden("CAB-THHN-12", alerta.cantidad_sugerida, 180_000)])
ordenes.approve("OC-0001"); ordenes.recibir("OC-0001")
print("Stock después de recibir la orden:", inventario.stock_de("CAB-THHN-12"))
for evento in ordenes.auditoria: print("  auditoría:", evento)

In [ ]:
import pandas as pd
from kuboerp.nomina import MotorNomina
from kuboerp.talento_humano import Tecnico, Vendedor, Gerente, Administrativo, Novedades

planta = [
    Tecnico("1072345678", "Julián Rojas", "Técnico electricista", "INDEFINIDO", 2_200_000),
    Vendedor("52987654", "Laura Gómez", "Asesora comercial", "FIJO", 1_750_905),
    Gerente("79123456", "Andrea Castro", "Gerente comercial", "INDEFINIDO", 12_000_000),
    Administrativo("1010203040", "Camila Díaz", "Auxiliar contable", "OBRA_LABOR", 2_000_000),
]
novedades = {"1072345678": Novedades(horas_extra_diurnas=10), "52987654": Novedades(ventas=10_000_000),
             "79123456": Novedades(bonificacion=9_000_000)}
motor = MotorNomina()
desprendibles = motor.liquidar_periodo(planta, novedades)
tabla = pd.DataFrame([{"Empleado": d.nombre, "Devengado": d.total_devengado, "Deducciones": d.total_deducciones,
                       "Neto": d.neto, "Costo empleador": d.costo_empleador} for d in desprendibles])
display(tabla.style.format({c: "${:,.0f}" for c in tabla.columns[1:]}))
print(MotorNomina.resumen(desprendibles))